# Wojtek's Week 6 exercise

In [ ]:
# Imports
import matplotlib.pyplot as plt
import random

import numpy as np
from tqdm.notebook import tqdm
from sklearn.feature_extraction.text import HashingVectorizer
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR

from pricer.items import Item
from pricer.evaluator import evaluate

In [ ]:
LITE_MODE = False
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

In [ ]:
# Reduce size of items
items = items[:1]

# Reduce size of train
train = random.sample(train, 300_000)

In [ ]:
print(len(val))
print(len(test))
print(len(train))
print(len(items))

## Proposed structure: NeuralNetwork (description → price)

**Goal:** Predict `item.price` from `item.summary` only, with a model under 100M parameters for local training.

**Pipeline:**
1. **Text → vector:** `HashingVectorizer(n_features=5000)` on `summary` (bag-of-words). Same as your existing setup; keeps input size fixed and works well for product text.
2. **Network:** MLP with residual blocks:
   - **Input layer:** Linear(5000 → 2048), LayerNorm, ReLU, Dropout.
   - **Residual blocks:** 5 blocks of two Linear(2048→2048) layers with LayerNorm and a skip connection (improves gradient flow and training stability).
   - **Output:** Linear(2048 → 1) for a single price prediction.

**Why this shape:** Bag-of-words captures term presence; the deep MLP learns nonlinear combinations that correlate with price. Hidden size 2048 and 4 blocks yield ~40–50M parameters (well under 100M), so training on CPU or a single GPU is feasible.

In [ ]:
# NeuralNetwork: description (summary) -> price, under 100M params
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn


class _ResidualBlock(nn.Module):
    """Single residual block: Linear -> LayerNorm -> ReLU -> Dropout -> Linear -> LayerNorm + skip -> ReLU."""
    def __init__(self, hidden_size: int, dropout_prob: float = 0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(x + self.block(x))


class NeuralNetwork(nn.Module):
    """
    Predict price from item summary only.
    Input: bag-of-words vector of shape (batch, 5000) from HashingVectorizer(5000) on summary.
    Output: (batch, 1) price prediction.
    """
    def __init__(
        self,
        input_size: int = 5000,
        hidden_size: int = 2048,
        num_residual_blocks: int = 5,
        dropout_prob: float = 0.2,
    ):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
        )
        self.blocks = nn.ModuleList([
            _ResidualBlock(hidden_size, dropout_prob) for _ in range(num_residual_blocks)
        ])
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.input_layer(x)
        for block in self.blocks:
            x = block(x)
        return self.output_layer(x)


# Example: build model and check parameter count (use with vectorizer output size)
_vectorizer = HashingVectorizer(n_features=5000, stop_words="english", binary=True)
_example_X = _vectorizer.fit_transform([items[0].summary])
_input_size = _example_X.shape[1]  # 5000
_nn1 = NeuralNetwork(input_size=_input_size)
_param_count = sum(p.numel() for p in _nn1.parameters() if p.requires_grad)
print(f"NeuralNetwork1 trainable parameters: {_param_count:,} (under 100M: {_param_count < 100_000_000})")

In [ ]:
DEVICE_OPT = "auto"  # change to "cpu" or "cuda" to force

if DEVICE_OPT == "auto":
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device(DEVICE_OPT)
print(f"Training on: {device}")

# Data: vectorize summaries, log-normalize prices
np.random.seed(42)
torch.manual_seed(42)
train_vec = HashingVectorizer(n_features=5000, stop_words="english", binary=True)
X_train_np = train_vec.fit_transform([item.summary for item in train])
X_val_np = train_vec.transform([item.summary for item in val])

y_train_np = np.array([float(item.price) for item in train], dtype=np.float32)
y_val_np = np.array([float(item.price) for item in val], dtype=np.float32)
y_train = torch.FloatTensor(y_train_np).unsqueeze(1)
y_val = torch.FloatTensor(y_val_np).unsqueeze(1)

# Log-scale targets for more stable regression
y_train_log = torch.log(y_train + 1)
y_val_log = torch.log(y_val + 1)
y_mean = y_train_log.mean()
y_std = y_train_log.std()
y_train_norm = (y_train_log - y_mean) / y_std
y_val_norm = (y_val_log - y_mean) / y_std

X_train_t = torch.FloatTensor(X_train_np.toarray())
X_val_t = torch.FloatTensor(X_val_np.toarray())

train_ds = TensorDataset(X_train_t, y_train_norm)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

# Model, loss, optimizer
input_size = X_train_t.shape[1]
model_nn1 = NeuralNetwork(input_size=input_size).to(device)
criterion = nn.L1Loss()
optimizer_nn1 = optim.AdamW(model_nn1.parameters(), lr=0.001, weight_decay=0.01)
scheduler_nn1 = CosineAnnealingLR(optimizer_nn1, T_max=10, eta_min=1e-5)

In [ ]:
# Training loop

EPOCHS = 6
history = {"train_loss": [], "val_loss": [], "val_mae": []}

for epoch in range(1, EPOCHS + 1):
    model_nn1.train()
    train_losses = []
    for batch_X, batch_y in tqdm(train_loader, desc=f"Epoch {epoch}"):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer_nn1.zero_grad()
        out = model_nn1(batch_X)
        loss = criterion(out, batch_y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_nn1.parameters(), max_norm=1.0)
        optimizer_nn1.step()
        train_losses.append(loss.item())
    scheduler_nn1.step()
    train_loss_epoch = np.mean(train_losses)
    history["train_loss"].append(train_loss_epoch)

    model_nn1.eval()
    with torch.no_grad():
        val_out = model_nn1(X_val_t.to(device))
        val_loss = criterion(val_out, y_val_norm.to(device)).item()
        val_pred_orig = torch.exp(val_out * y_std + y_mean) - 1
        val_mae = torch.abs(val_pred_orig - y_val.to(device)).mean().item()
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)
    print(f"Epoch {epoch}/{EPOCHS}  train_loss: {train_loss_epoch:.4f}  val_loss: {val_loss:.4f}  val_MAE: ${val_mae:.2f}")

In [ ]:
# Plot train/val loss and MAE after training
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(epochs_range, history["train_loss"], label="Train loss", color="C0")
ax1.plot(epochs_range, history["val_loss"], label="Val loss", color="C1")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Train vs validation loss")
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax2 = plt.subplots(figsize=(8, 4))
ax2.plot(epochs_range, history["val_mae"], color="C2")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("MAE ($)")
ax2.set_title("Validation MAE")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def wojtek_neural_network(item):
    model_nn1.eval()
    with torch.no_grad():
        # Vectorize summary, using the same vectorizer as training set
        vector = train_vec.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray()).to(device)
        # Predict normalized log price
        out = model_nn1(vector)
        # De-normalize and convert log prediction back to dollar value
        result = torch.exp(out * y_std + y_mean) - 1
        result = result.item()
    return max(0, result)

In [ ]:
evaluate(wojtek_neural_network, test, size=1000)